# Fine-tuning LoRA — Modèle médical expérimental (TechCorp)

A exécuter sur **Google Colab** (runtime GPU : `Runtime > Change runtime type > GPU`, T4 suffit).

- Base : `microsoft/Phi-3.5-mini-instruct`
- Dataset : `datasets/cleaned/medical_dataset_sample.json` (5000 dialogues patient/médecin nettoyés, voir `datasets/MEDICAL_DATA_QUALITY_REPORT.md`)
- Objectif : POC expérimental — **pas un déploiement production** (voir `medical_project/Readme.md`).

A la fin du notebook : l'adaptateur LoRA entraîné + les métriques (loss/epochs) sont sauvegardés. Pensez à partager le **lien Colab** et les métriques avec le reste de l'équipe.

In [ ]:
!pip install -q transformers>=4.45.0 peft>=0.12.0 accelerate>=0.34.0 bitsandbytes>=0.43.0 datasets>=2.20.0

## 1. Récupérer le dataset

Deux options : cloner le repo (nécessite `git-lfs`), ou uploader `medical_dataset_sample.json` directement via le panneau Fichiers de Colab.

In [ ]:
# Option A : clone du repo (remplacez l'URL par celle de votre remote)
!apt-get -qq install git-lfs
!git lfs install
!git clone https://github.com/BertaudNathan/hackathon_ynov_2026_grp_5.git repo
!cd repo && git lfs pull
DATASET_PATH = "repo/datasets/cleaned/medical_dataset_sample.json"

In [ ]:
# Option B (si l'option A échoue) : uploader medical_dataset_sample.json manuellement puis décommenter
# from google.colab import files
# uploaded = files.upload()
# DATASET_PATH = "medical_dataset_sample.json"

In [ ]:
import json

with open(DATASET_PATH, encoding="utf-8") as f:
    raw_data = json.load(f)

print(f"{len(raw_data)} exemples charg\u00e9s")
print(raw_data[0])

## 2. Chargement du modèle de base (4-bit) + configuration LoRA

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training

BASE_MODEL = "microsoft/Phi-3.5-mini-instruct"

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quantization_config,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.float16,
)
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["qkv_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 3. Préparation du dataset (format chat Phi-3)

In [ ]:
from datasets import Dataset

def to_text(example):
    text = f"<|user|>\n{example['instruction']}<|end|>\n<|assistant|>\n{example['output']}<|end|>"
    return {"text": text}

texts = [to_text(x) for x in raw_data]
hf_dataset = Dataset.from_list(texts)

def tokenize_function(examples):
    tokenized = tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=512,
    )
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

tokenized_dataset = hf_dataset.map(tokenize_function, batched=True, remove_columns=["text"])
tokenized_dataset

## 4. Entraînement

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

training_args = TrainingArguments(
    output_dir="./phi35_medical_lora_checkpoints",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    warmup_steps=50,
    logging_steps=25,
    save_steps=500,
    save_total_limit=2,
    remove_unused_columns=False,
    fp16=True,
    report_to="none",
)

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
)

train_result = trainer.train()

## 5. Sauvegarde du modèle + métriques (loss / epochs)

A partager avec l'équipe : le fichier `training_metrics.json`, le graphe de loss, et le lien de ce notebook Colab.

In [ ]:
import json
import matplotlib.pyplot as plt

trainer.save_model("./phi35_medical_lora")
tokenizer.save_pretrained("./phi35_medical_lora")

log_history = trainer.state.log_history
with open("training_metrics.json", "w") as f:
    json.dump(log_history, f, indent=2)

losses = [(e["epoch"], e["loss"]) for e in log_history if "loss" in e]
if losses:
    epochs, loss_values = zip(*losses)
    plt.figure(figsize=(8, 4))
    plt.plot(epochs, loss_values)
    plt.xlabel("Epoch")
    plt.ylabel("Training loss")
    plt.title("Phi-3.5 Medical LoRA — training loss")
    plt.savefig("training_loss.png")
    plt.show()
    print(f"Loss initiale: {loss_values[0]:.4f} -> Loss finale: {loss_values[-1]:.4f}")
    print(f"Nombre d'epochs: {epochs[-1]:.2f}")

## 6. Test rapide (validation conversationnelle)

⚠️ Modèle expérimental — les réponses ne constituent pas un avis médical (voir `medical_project/Readme.md`).

In [ ]:
test_questions = [
    "I have had a headache and mild fever for two days, what could it be?",
    "What are common side effects of ibuprofen?",
    "How much water should I drink daily?",
]

model.eval()
for q in test_questions:
    formatted = f"<|user|>\n{q}<|end|>\n<|assistant|>\n"
    inputs = tokenizer(formatted, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=150,
            temperature=0.7,
            do_sample=True,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
        )
    response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    print(f"Q: {q}\nR: {response}\n{'-'*60}")

## 7. Récupération des livrables

Téléchargez `./phi35_medical_lora/` (adaptateur), `training_metrics.json` et `training_loss.png`, puis :
- placez l'adaptateur dans `medical_project/phi35_medical_lora/` du repo
- partagez le lien de ce notebook Colab + les métriques (loss, epochs) avec l'équipe pour le livrable final

In [ ]:
!zip -r phi35_medical_lora.zip phi35_medical_lora training_metrics.json training_loss.png
from google.colab import files
files.download("phi35_medical_lora.zip")